In [ ]:
# ==============================================================================
# CELL 0: HARDWARE, ENVIRONMENT & WHEELHOUSE INSTALLATION
# ==============================================================================
import glob
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'

_CANDS = [
    Path('/kaggle/input/gemma-4-developer-agent-wheelhouse'),
    Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
]
WHEELHOUSE_DIR = next((p for p in _CANDS if p.exists() and list(p.glob('*.whl'))), None)
if WHEELHOUSE_DIR is None:
    _found = sorted(Path('/kaggle/input').rglob('swegemma*.whl'))
    assert _found, 'no swegemma wheel anywhere under /kaggle/input'
    WHEELHOUSE_DIR = _found[0].parent
print('USING WHEELHOUSE:', WHEELHOUSE_DIR)

for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')


In [ ]:
# ==============================================================================
# CELL 1: EXTRACT SUBMISSION & MOUNT FINE-TUNED LORA ADAPTER
# ==============================================================================
import base64
import io
import os
import shutil
import zipfile
from pathlib import Path

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
AGENT_DIR = WORKING_DIR / 'submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
AGENT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Unpack declarative agent assets
B64_SUBMISSION = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(B64_SUBMISSION))) as zf:
    zf.extractall(AGENT_DIR)
print(f'Extracted declarative submission assets to {AGENT_DIR}')

# 2. Locate and mount LoRA adapter
adapter_cands = [
    Path('/kaggle/input/gemma4-lora-adapter'),
    Path('/kaggle/input/datasets/francisclyap/gemma4-lora-adapter')
]
ADAPTER_SRC = next((p for p in adapter_cands if p.exists() and (p / 'adapter_model.safetensors').exists()), None)
if ADAPTER_SRC is None:
    found_tensors = list(Path('/kaggle/input').rglob('adapter_model.safetensors'))
    assert found_tensors, 'adapter_model.safetensors not found under /kaggle/input!'
    ADAPTER_SRC = found_tensors[0].parent
print('USING LORA ADAPTER SOURCE:', ADAPTER_SRC)

LORA_DEST = AGENT_DIR / 'adapters' / 'main_lora'
LORA_DEST.mkdir(parents=True, exist_ok=True)
for item in ADAPTER_SRC.iterdir():
    if item.is_file() and not item.name.startswith('.'):
        shutil.copy2(item, LORA_DEST / item.name)

print('LoRA adapter mounted at:', LORA_DEST)
for p in sorted(AGENT_DIR.rglob('*')):
    if p.is_file():
        print(f'  - {p.relative_to(AGENT_DIR)}: {p.stat().st_size:,} bytes')


In [ ]:
# ==============================================================================
# CELL 2: START OFFLINE VLLM SERVER (WITH LORA ADAPTER ENABLED)
# ==============================================================================
import litellm
import torch
from pathlib import Path
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

litellm.drop_params = True

TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
_MP = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
if not _MP.exists():
    _f = sorted(Path('/kaggle/input').rglob('gemma-4-31b-it-qat-w4a16-ct'))
    print('model fallback search:', _f[:3])
    _MP = _f[0] if _f else _MP
MODEL_PATH = _MP
print('USING MODEL:', MODEL_PATH, MODEL_PATH.exists())

declared_model = validate_single_declared_model(AGENT_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
print(f'Discovered Adapters: {adapters}')

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
print(f'Detected GPUs: {gpu_count}, Tensor Parallel Size: {tp_size}')

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=bool(adapters),
    max_loras=8 if adapters else 0,
    max_lora_rank=128 if adapters else 0,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

models = server_instance.create_model_registry(
    aliases=[declared_model, TARGET_MODEL_NAME],
    model_prefix='openai/',
    api_key='EMPTY',
)
print('Model registry created successfully.')


In [ ]:
# ==============================================================================
# CELL 3: RUN CANARY EVALUATION ON TWO TARGET FAILED TASKS
# ==============================================================================
import asyncio
import concurrent.futures
import json
import os
import shutil
from pathlib import Path
import pandas as pd
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator
from swegemma.models import load_tasks

def run_sync(coro_or_fn, *args, **kwargs):
    fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')

# Two high-signal near-miss tasks:
# 1. fastapi_14479: small assertion error string mismatch
# 2. requests_7205: _parse_content_type_header boolean flag handling
TARGET_IDS = ['fastapi_14479', 'requests_7205']
canary_tasks = [t for t in tasks if t.instance_id in TARGET_IDS]
print(f'Target Tasks ({len(canary_tasks)}): {[t.instance_id for t in canary_tasks]}')

limits, gen_constraints = build_submission_limits()
RESULTS_DIR = WORKING_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
JSONL_PATH = RESULTS_DIR / 'task_results.jsonl'

eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=RESULTS_DIR,
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=60,
    max_time_minutes=4.0,
    max_tool_calls=30,
    max_turns=50,
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15, overlap_size=2,
        token_threshold=14336, event_retention_size=5),
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    wheels_dir=DATA_DIR / 'wheels',
    verbose=True,
)
evaluator = Evaluator(eval_config)

resolved_count = 0
total_evaluated = 0

print('\nStarting canary evaluation of fine-tuned Gemma 4 LoRA agent...\n', flush=True)

for idx, task in enumerate(canary_tasks, start=1):
    print(f'[{idx}/{len(canary_tasks)}] Evaluating {task.instance_id} ({task.repo})...', flush=True)
    try:
        result = run_sync(evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(canary_tasks))
        is_resolved = bool(result.resolved)
        if is_resolved:
            resolved_count += 1
        total_evaluated += 1

        record = {
            'task_id': task.instance_id,
            'repo': task.repo,
            'resolved': is_resolved,
            'test_exit_code': result.test_exit_code,
            'tool_calls': result.tool_calls,
            'duration_seconds': round(result.duration_seconds, 2),
            'patch_length': len(result.agent_patch or ''),
            'patch': result.agent_patch or '',
        }

        with open(JSONL_PATH, 'a', encoding='utf-8') as jf:
            jf.write(json.dumps(record) + '\n')

        print(
            f'  -> resolved={is_resolved} (Running Score: {resolved_count}/{total_evaluated}), '
            f'exit={result.test_exit_code}, patch={record["patch_length"]}ch, calls={result.tool_calls}, {result.duration_seconds:.1f}s',
            flush=True
        )
    except Exception as e:
        print(f'  -> ERROR evaluating {task.instance_id}: {e}', flush=True)
    finally:
        for sb_dir in Path('/tmp').glob('swegemma_sandbox_*'):
            try:
                shutil.rmtree(sb_dir, ignore_errors=True)
            except Exception:
                pass

print(f'\n=================== CANARY RUN COMPLETE ===================', flush=True)
print(f'Canary Score: {resolved_count}/{total_evaluated} resolved.', flush=True)


In [ ]:
# ==============================================================================
# CELL 4: TRACE AND PATCH INSPECTION
# ==============================================================================
import json
from pathlib import Path

print('=== GENERATED PATCHES ===\n')
if JSONL_PATH.exists():
    with open(JSONL_PATH) as f:
        for line in f:
            rec = json.loads(line)
            print(f"Task: {rec['task_id']} | Resolved: {rec['resolved']} | Exit Code: {rec['test_exit_code']}")
            print(f"Patch ({rec['patch_length']} chars):\n")
            print(rec.get('patch', '')[:1000])
            print('-' * 60)

print('\nCanary evaluation complete.')
